# MixNet — one-click Colab run

Runtime → Change runtime type → **GPU**, then Runtime → **Run all**. Edit the config cell first if needed. Re-running skips models that already have results.

In [ ]:
MODELS = ["v1", "v2", "v3"]
EPOCHS = 60
PATIENCE = 15
NOISE = 0.0
SEEDS = 30
REF = "main"
FORCE = False
SKIP_TESTS = False
IGNORE_TEST_FAILURES = False
ALLOW_CPU = False
REPO_URL = "https://github.com/rounakrkr/nmr_deconvolution.git"
WORKDIR = "/content/nmr_deconvolution"

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from collections import deque

BATCH_SIZE = {"v1": 4, "v2": 4, "v3": 2}
PIP_PACKAGES = ["scipy", "pandas", "pyyaml", "tqdm", "matplotlib", "pytest"]
T_START = time.time()
FAILED = []


def run(cmd, cwd=None, tail=80):
    print("\n$ " + " ".join(cmd), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = deque(maxlen=tail)
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        lines.append(line)
    proc.wait()
    return proc.returncode, "".join(lines)


def banner(text):
    print("\n" + "=" * 78 + f"\n  {text}\n" + "=" * 78, flush=True)


def prepare_repo():
    if not os.path.isdir(os.path.join(WORKDIR, ".git")):
        code, _ = run(["git", "clone", REPO_URL, WORKDIR])
        if code != 0:
            raise RuntimeError("git clone failed")
    run(["git", "fetch", "origin"], cwd=WORKDIR)
    for candidate in (REF, "main"):
        code, _ = run(["git", "checkout", "-q", "-B", candidate, f"origin/{candidate}"], cwd=WORKDIR)
        if code == 0:
            print(f"using ref: {candidate}")
            return
    raise RuntimeError(f"could not check out '{REF}' or 'main'")


def verify_split_fix():
    with open(os.path.join(WORKDIR, "train.py")) as f:
        text = f.read()
    if '"--n-val-compounds", type=int, default=4' in text:
        raise RuntimeError("this ref still has the old split defaults (val pool of 4 compounds) and crashes the sampler. Set REF to a ref that contains fix/split-defaults.")


def check_gpu():
    import torch

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        return
    if not ALLOW_CPU:
        raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU, or set ALLOW_CPU = True.")
    print("WARNING: running on CPU, this will be very slow.")


def train_model(model):
    if model not in MODELS:
        print(f"{model}: not in MODELS, skipping")
        return
    result_path = os.path.join(WORKDIR, "results", f"{model}_heldout_benchmark.json")
    if os.path.exists(result_path) and not FORCE:
        print(f"{model}: results already present, skipping (set FORCE = True to retrain)")
        return
    t0 = time.time()
    batch = BATCH_SIZE[model]
    while True:
        cmd = [
            sys.executable, "train.py", "--model", model,
            "--epochs", str(EPOCHS), "--patience", str(PATIENCE),
            "--batch-size", str(batch), "--noise", str(NOISE),
        ]
        code, out = run(cmd, cwd=WORKDIR)
        if code == 0:
            break
        if "out of memory" in out.lower() and batch > 1:
            batch = max(1, batch // 2)
            print(f"{model}: CUDA out of memory, retrying with batch size {batch}")
            continue
        FAILED.append(model)
        print(f"{model}: training failed (exit code {code})")
        return
    print(f"{model} finished in {(time.time() - t0) / 60:.1f} min")


def load_json(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def cell(r):
    return f"{r['mean']:.4f} ± {r['ci95']:.4f}"


def print_summary():
    banner("SUMMARY (Pearson correlation, Hungarian-matched; higher is better)")
    heldout = {m: load_json(os.path.join(WORKDIR, "results", f"{m}_heldout_benchmark.json")) for m in MODELS}
    heldout = {m: d for m, d in heldout.items() if d}
    if heldout:
        print("\nHeld-out compounds (never seen in training)")
        print(f"{'method':<22}{'matched corr':>20}{'fixed-slot corr':>20}{'best epoch':>12}")
        for m, d in heldout.items():
            res = d["results"]
            print(f"{'MixNet ' + m.upper():<22}{cell(res['model_matched']):>20}{cell(res['model_slot']):>20}{d['best_epoch']:>12}")
        base = next(iter(heldout.values()))["results"]
        for name in ("nmf", "mean_spectrum"):
            print(f"{name:<22}{cell(base[name]):>20}")
    blind = load_json(os.path.join(WORKDIR, "results", "blind_test.json"))
    if blind:
        rep = blind["report"]
        print(f"\nNovel compounds blind test ({blind['seeds']} seeds, noise {blind['noise']})")
        print(f"{'method':<22}{'matched corr':>20}{'fixed-slot corr':>20}")
        for name, r in rep.items():
            if name == "baselines":
                for b in ("nmf", "mean_spectrum"):
                    print(f"{b:<22}{cell(r[b]):>20}")
            else:
                print(f"{'MixNet ' + name.upper():<22}{cell(r['model_matched']):>20}{cell(r['model_slot']):>20}")


def package_outputs():
    base = "/content" if os.path.isdir("/content") else os.getcwd()
    zip_path = os.path.join(base, "mixnet_outputs.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for folder in ("results", "checkpoints"):
            for dirpath, _, files in os.walk(os.path.join(WORKDIR, folder)):
                for name in files:
                    full = os.path.join(dirpath, name)
                    z.write(full, os.path.relpath(full, WORKDIR))
    print(f"\npackaged: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
    drive = "/content/drive/MyDrive"
    if os.path.isdir(drive):
        shutil.copy(zip_path, drive)
        print(f"copied to {drive}")
    try:
        from google.colab import files

        files.download(zip_path)
    except Exception:
        pass

In [ ]:
banner("setup")
check_gpu()
prepare_repo()
verify_split_fix()
run([sys.executable, "-m", "pip", "install", "-q"] + PIP_PACKAGES)

In [ ]:
banner("tests")
if SKIP_TESTS:
    print("skipped")
else:
    code, _ = run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=WORKDIR)
    if code != 0 and not IGNORE_TEST_FAILURES:
        raise RuntimeError("pytest failed. Fix the failure above, or set IGNORE_TEST_FAILURES = True.")

In [ ]:
banner("train V1")
train_model("v1")

In [ ]:
banner("train V2")
train_model("v2")

In [ ]:
banner("train V3")
train_model("v3")

In [ ]:
banner("blind test")
checkpoints = [
    os.path.join("checkpoints", f"best_{m}.pth")
    for m in MODELS
    if os.path.exists(os.path.join(WORKDIR, "checkpoints", f"best_{m}.pth"))
]
cmd = [sys.executable, "blind_test.py", "--seeds", str(SEEDS), "--noise", str(NOISE)]
if checkpoints:
    cmd += ["--checkpoints"] + checkpoints
code, _ = run(cmd, cwd=WORKDIR)
if code != 0:
    FAILED.append("blind_test")

In [ ]:
print_summary()
package_outputs()
print(f"\ntotal time: {(time.time() - T_START) / 60:.1f} min")
if FAILED:
    raise RuntimeError(f"failed steps: {', '.join(FAILED)}")